# Amazon Food Reviews Sentiment Analysis Classification with BERT

This notebook demonstrates how to fine-tune a BERT model for sentiment classification on the Amazon Food Reviews dataset.

## Possible business use cases for a fine-tuned BERT model
- **Marketing Insights:** Detect customer emotions in real-time to adjust marketing strategies.
- **Product & Service Feedback:** Analyze customer reviews to improve products or services.
- **Automatic Review Filtering:** Identify and flag negative or inappropriate comments for quick action.
- **Competitor Comparison:** Track how customers feel about your brand versus competitors.
- **Brand Reputation Monitoring:** Keep an eye on customer sentiment and respond to issues quickly.


## Step 1: Install Libraries


In [1]:
!pip install --upgrade transformers --upgrade httpx fsspec numpy datasets evaluate torch tqdm textaugment textblob==0.17.1 googletrans==4.0.0-rc1 nltk

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)


In [2]:
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
import numpy as np
from transformers import BertForSequenceClassification
from transformers import Trainer, TrainingArguments, EvalPrediction
import evaluate
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix
from textaugment import EDA
from tqdm import tqdm
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
tqdm.pandas()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


## Step 2: Load Dataset


In [3]:
# ✅ Load Dataset
df = pd.read_csv("Reviews.csv")

# ✅ Keep only necessary columns
df = df[["Score", "Text"]].copy()

# ✅ Convert Score to Sentiment Labels
def label_sentiment(score):
    if score >= 4:
        return 2  # Positive
    elif score == 3:
        return 1  # Neutral
    else:
        return 0  # Negative

df["label"] = df["Score"].apply(label_sentiment)
df = df.drop(columns=["Score"])  # Drop original Score column

## Step 3: Data Augmentation


In [4]:
eda_aug = EDA()

tqdm.pandas(desc="Applying EDA Augmentation")

# ✅ Use EDA synonym replacement for augmentation
df["Text_augmented"] = df["Text"].progress_apply(
    lambda x: eda_aug.synonym_replacement(x, n=1) if np.random.rand() < 0.2 else x
)

Applying EDA Augmentation: 100%|██████████| 568454/568454 [00:41<00:00, 13787.62it/s]


## Step 4: Data Balancing & Preprocessing


In [5]:
# ✅ Get class distribution
class_counts = df['label'].value_counts()
min_samples = min(class_counts)  # Get the smallest class size

# ✅ Balance the dataset by sampling equal instances from each class
balanced_df = df.groupby('label').apply(lambda x: x.sample(min_samples, random_state=42)).reset_index(drop=True)

# ✅ Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(balanced_df)

# ✅ Split dataset into train and test (80-20 split)
split = dataset.train_test_split(test_size=0.2)

# ✅ Select 5000 samples for training and 1000 for testing
train_dataset = split['train'].shuffle(seed=42).select(range(5000))
test_dataset  = split['test'].shuffle(seed=42).select(range(1000))


<ipython-input-5-eb945002f392>:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = df.groupby("label").apply(lambda x: x.sample(min_samples, random_state=42)).reset_index(drop=True)


## Step 5: Tokenization & Data Encoding


In [6]:
# ✅ Load BERT Tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# ✅ Tokenize dataset — use augmented text so EDA augmentation takes effect
def tokenize_function(examples):
    tokens = tokenizer(examples['Text_augmented'], padding='max_length', truncation=True)
    tokens['labels'] = examples['label']  # Ensure labels are included
    return tokens

# ✅ Apply tokenization
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset  = test_dataset.map(tokenize_function, batched=True)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## Step 6: DataLoader Creation & Batch Processing


In [7]:
# ✅ Define custom collate function for DataLoader
def collate_fn(batch):
    return {
        "input_ids": torch.tensor([item["input_ids"] for item in batch], dtype=torch.long),
        "attention_mask": torch.tensor([item["attention_mask"] for item in batch], dtype=torch.long),
        "labels": torch.tensor([item["labels"] for item in batch], dtype=torch.long) if "labels" in batch[0] else None,
    }

# ✅ Set batch size
batch_size = 16

# ✅ Create DataLoaders
train_dataloader = DataLoader(tokenized_train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_dataloader = DataLoader(tokenized_test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)


## Step 7: Model Setup & Configuration


In [8]:
# ✅ Set device (CUDA if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ✅ Load BERT Model for 3-class classification
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=3, hidden_dropout_prob=0.4
).to(device)

# ✅ Compute class weights to handle class imbalance
raw_counts   = [240 + 82 + 15, 46 + 256 + 39, 8 + 45 + 269]  # approximate; update with real counts if needed
total_samples = sum(raw_counts)
class_weights = torch.tensor(
    [total_samples / c for c in raw_counts], dtype=torch.float
).to(device)

epochs = 4  # keep consistent with TrainingArguments below

# ✅ Custom Trainer that applies the weighted loss so class imbalance is addressed
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = CrossEntropyLoss(weight=class_weights)
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Step 8: Train the Model


In [9]:
# ✅ Load evaluation metric (accuracy)
accuracy_metric = evaluate.load('accuracy')

# ✅ Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=epochs,
    weight_decay=0.05,
    learning_rate=3e-5,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_dir='./logs',
    logging_steps=10,
    save_total_limit=1,
    report_to='none',
)

# ✅ Function to compute evaluation metrics (accuracy)
def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# ✅ Initialize WeightedTrainer (uses class-weighted loss defined in Step 6)
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    processing_class=tokenizer,  # replaces deprecated tokenizer= argument
    compute_metrics=compute_metrics,
)

# ✅ Train the model
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,0.723800,0.632247,0.746000
2,0.693800,0.676470,0.733000
3,0.570900,0.635153,0.752000
4,0.502700,0.706098,0.748000


TrainOutput(global_step=2500, training_loss=0.6213705780982971, metrics={'train_runtime': 2037.6524, 'train_samples_per_second': 9.815, 'train_steps_per_second': 1.227, 'total_flos': 5262268354560000.0, 'train_loss': 0.6213705780982971, 'epoch': 4.0})

## Step 9: Evaluate


In [10]:
# ✅ Define Evaluation Function
def evaluate_model(model, dataloader):
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits if hasattr(outputs, "logits") else outputs

            predictions = torch.argmax(logits, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predictions.cpu().numpy())

    print("Detailed Classification Report:")
    print(classification_report(all_labels, all_preds))
    print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))

# ✅ Run Evaluation (Only Once)
evaluate_model(model, test_dataloader)

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.81      0.75       352
           1       0.72      0.57      0.64       338
           2       0.82      0.87      0.84       310

    accuracy                           0.75      1000
   macro avg       0.75      0.75      0.74      1000
weighted avg       0.74      0.75      0.74      1000

Confusion Matrix:
 [[284  48  20]
 [106 193  39]
 [ 13  28 269]]
